In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Symplectic Integrators

Last time we ended on a cliffhanger.  We taught our integrators to handle Newton's second law, pointed them at the simplest system in all of physics, the harmonic oscillator, and watched them break one of the most sacred laws in physics: conservation of energy.  Euler's method pumped in energy so fast that by $t=1000$ it had grown by a factor of $10^{43}$.  RK2 was *way* better, but it still leaked a little bit every single step, and the leak never stopped.

Let's fix that problem.

Amazingly, the fix is basically swapping the order of two lines of code!  But *why* that works is the fun, interesting part.  It will lead us to a beautiful idea about what an integrator does to *phase space*.

## Our Tools So Far

Here's everything we need from last time, copied over without changes.  (Skim past them if you remember them!)  The only new thing is that the `Pendulum` now knows its energy too.

In [ ]:
class FirstOrderODE:
    '''
    A first-order ODE specifies the rate of change of a state y(t)
    in terms of
     - the state itself, y
     - the time, t.

    The generic form for an ODE is

    dy
    --  = f(t, y)
    dt

    '''

    def f(self, t, y):
        raise NotImplementedError('The generic ODE has an unspecified slope function f.')

    def exact_solution(self, t, *args):
        raise NotImplementedError('The exact solution is not known (or at least not provided).')

In [ ]:
class SecondOrderODE(FirstOrderODE):
    '''
    A second-order ODE specifies the acceleration

    d^2x
    ----  = a(t, x, v)
    dt^2

    where v = dx/dt is the velocity.  Treating the state as y = (x, v) turns it into
    a first-order ODE,

    d  ( x )   (    v     )
    -- (   ) = (          )
    dt ( v )   ( a(t,x,v) )

    so a subclass only needs to provide the acceleration.
    '''

    def acceleration(self, t, x, v):
        raise NotImplementedError('The generic second-order ODE has an unspecified acceleration.')

    def f(self, t, y):
        # Unpack the state into its two components,
        x, v = y
        # and pack their rates of change back up the same way.
        return np.array([v, self.acceleration(t, x, v)])

In [ ]:
class HarmonicOscillator(SecondOrderODE):
    '''
    A harmonic oscillator has acceleration a = -ω^2 x.
    '''

    def __init__(self, omega):
        self.omega = omega

    def acceleration(self, t, x, v):
        return -self.omega**2 * x

    def exact_solution(self, t, x_0, v_0):
        x =  x_0 * np.cos(self.omega * t) + v_0 / self.omega * np.sin(self.omega * t)
        v = -x_0 * self.omega * np.sin(self.omega * t) + v_0 * np.cos(self.omega * t)
        return np.array([x, v])

    def energy(self, x, v):
        # The energy per unit mass
        return 0.5 * v**2 + 0.5 * self.omega**2 * x**2

In [ ]:
class Pendulum(SecondOrderODE):
    '''
    A simple pendulum has angular acceleration a = -ω^2 sin(θ), where ω^2 = g/ℓ.
    We use x for the angle θ and v for the angular velocity.
    '''

    def __init__(self, omega):
        self.omega = omega

    def acceleration(self, t, x, v):
        return -self.omega**2 * np.sin(x)

    def energy(self, x, v):
        # The energy, divided by m ℓ^2 to keep things tidy
        return 0.5 * v**2 + self.omega**2 * (1 - np.cos(x))

In [ ]:
class FirstOrderODEIntegrator:
    '''
    An integrator is an algorithm for producing numerical solutions to ODEs.
    '''

    def __init__(self, ODE):

        self.ODE = ODE

    def starting_from(self, t_0, y_0):
        raise NotImplementedError('No integration algorithm has been specified.')

In [ ]:
class EulerMethod(FirstOrderODEIntegrator):
    '''
    Euler's method is based on the Taylor series approximation

    y(t + Δt) = y(t) + y'(t) Δt + (higher-order terms with more powers of Δt which we neglect)

    Given a starting location on the solution (t_0, y_0) we can repeatedly say
    - the next t will be t_0 + Δt
    - the next y will be y_0 + Δy, and using the Taylor series Δy = y'(t) Δt

    This produces the next point on the solution, and we can begin again.
    '''

    def __init__(self, ODE, Delta_t, steps):

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):

        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array to store the y values we compute.
        # 
        # Previously we had
        #
        # y = np.zeros_like(t, dtype=float)
        #
        # but this only has room for one number at each time.
        y = np.zeros((self.steps+1,) + np.shape(y_0))   # room for a whole state at each time

        # The user picked a starting y value
        y[0] = y_0

        # and we can step through the Euler procedure over and over again.
        for step in range(self.steps):
            slope = self.ODE.f(t[step], y[step])
            y[step+1] = y[step] + slope * self.Delta_t

        return t, y

In [ ]:
class RungeKutta2(FirstOrderODEIntegrator):

    def __init__(self, ODE, alpha, Delta_t, steps):

        # Conditions for matching the Taylor expansion to 2nd order
        self.alpha = alpha
        self.b = 1/(2*self.alpha)
        self.a = 1 - self.b

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):
        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array to store the y states we compute.
        y = np.zeros((self.steps+1,) + np.shape(y_0))

        # The user picked a starting y value
        y[0] = y_0

        # and we can step through the RK2 procedure over and over again.
        for step in range(self.steps):
            k_1 = self.ODE.f(t[step], y[step])
            k_2 = self.ODE.f(t[step] + self.alpha * self.Delta_t, y[step] + self.alpha * k_1 * self.Delta_t)

            y[step+1] = y[step] + (self.a * k_1 + self.b * k_2) * self.Delta_t

        return t, y

# Where We Left Off

Let's remind ourselves of the problem.  We'll release the oscillator from $x=1$ at rest, take big steps of $\Delta t = 0.1$, and run for a long time, all the way to $t=1000$.  That's about 160 swings back and forth.

In [ ]:
oscillator = HarmonicOscillator(1.0)
y_0 = np.array([1.0, 0.0])

Delta_t = 0.1
steps = 10000

t, euler_y = EulerMethod(oscillator, Delta_t, steps).starting_from(0, y_0)
t, rk2_y   = RungeKutta2(oscillator, 0.5, Delta_t, steps).starting_from(0, y_0)

euler_energy = oscillator.energy(euler_y[:, 0], euler_y[:, 1])
rk2_energy   = oscillator.energy(rk2_y[:, 0],   rk2_y[:, 1])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(t, euler_energy / euler_energy[0], color='C0', label='Euler')
axes[0].plot(t, rk2_energy / rk2_energy[0], color='C1', label='RK2')
axes[0].set_yscale('log')

axes[1].plot(t, rk2_energy / rk2_energy[0], color='C1', label='RK2')

for axis in axes:
    axis.set_xlabel('t')
    axis.set_ylabel('E(t) / E(0)')
    axis.legend()

Euler's energy explodes, and RK2's creeps steadily upward.  We saw last time that shrinking $\Delta t$ only slows these leaks down; it never stops them.  So let's try something different.

# Swap the Order

Here's Euler's method written out for our two-number state $(x, v)$:
\begin{align}
    x_{n+1} &= x_n + v_n\, \Delta t
    \\
    v_{n+1} &= v_n + a(t_n, x_n)\, \Delta t.
\end{align}
Both updates use only the *old* values, $x_n$ and $v_n$.

Now here's a teeny change.  First update the velocity, using the force at the old position.  *Then* update the position, but use the *new* velocity you just computed:
\begin{align}
    v_{n+1} &= v_n + a(t_n, x_n)\, \Delta t  && \text{(kick)}
    \\
    x_{n+1} &= x_n + v_{n+1}\, \Delta t      && \text{(drift)}
\end{align}
People call the velocity update a *kick* (the force gives the velocity a little shove) and the position update a *drift* (the object coasts along at its new velocity).

This method goes by a bunch of names: the *symplectic Euler*, *semi-implicit Euler*, and *Euler-Cromer* method.  We'll find out what *symplectic* means soon.  It's still a first-order method (we'll check that later), so at first glance there's no reason to expect it to be any better than plain Euler.  Let's try it anyway!

Notice that this method needs the acceleration all by itself, not bundled up inside `f`.  That's fine: every `SecondOrderODE` has an `acceleration` method.  (Remember the idea of "duck typing": anything with an `acceleration` method will work.)

In [ ]:
class SymplecticEuler(FirstOrderODEIntegrator):
    '''
    Symplectic Euler first kicks the velocity using the force at the old position,

    v_{n+1} = v_n + a(t_n, x_n) Δt

    and then drifts the position using the *new* velocity,

    x_{n+1} = x_n + v_{n+1} Δt.

    It needs the ODE's acceleration, so it only works on second-order ODEs.
    '''

    def __init__(self, ODE, Delta_t, steps):

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):

        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array with room for a whole state at each time.
        y = np.zeros((self.steps+1,) + np.shape(y_0))

        # The user picked a starting state
        y[0] = y_0

        for step in range(self.steps):
            x, v = y[step]
            v_new = v + self.ODE.acceleration(t[step], x, v) * self.Delta_t   # kick
            x_new = x + v_new * self.Delta_t                                  # drift
            y[step+1] = np.array([x_new, v_new])

        return t, y

In [ ]:
t, symplectic_y = SymplecticEuler(oscillator, Delta_t, steps).starting_from(0, y_0)
symplectic_energy = oscillator.energy(symplectic_y[:, 0], symplectic_y[:, 1])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Left: phase space for the first ten time units
early = round(10 / Delta_t) + 1   # the number of steps with t ≤ 10
exact = oscillator.exact_solution(t[:early], 1.0, 0.0)

axes[0].plot(euler_y[:early, 0],      euler_y[:early, 1],      color='C0', label='Euler')
axes[0].plot(rk2_y[:early, 0],        rk2_y[:early, 1],        color='C1', label='RK2')
axes[0].plot(symplectic_y[:early, 0], symplectic_y[:early, 1], color='C2', label='symplectic Euler')
axes[0].plot(exact[0], exact[1], color='black', linestyle='dashed', label='exact')
axes[0].set_aspect('equal')
axes[0].set_xlabel('x')
axes[0].set_ylabel('v')
axes[0].set_title('t ≤ 10')
axes[0].legend()

# Middle: the energy over the first few periods
axes[1].plot(t[:early], rk2_energy[:early] / rk2_energy[0], color='C1', label='RK2')
axes[1].plot(t[:early], symplectic_energy[:early] / symplectic_energy[0], color='C2', label='symplectic Euler')
axes[1].set_xlabel('t')
axes[1].set_ylabel('E(t) / E(0)')
axes[1].set_title('Short-time behavior (t ≤ 10)')
axes[1].legend()

# Right: the energy over the whole long run
axes[2].plot(t, rk2_energy / rk2_energy[0], color='C1', label='RK2')
axes[2].plot(t, symplectic_energy / symplectic_energy[0], color='C2', label='symplectic Euler')
axes[2].set_xlabel('t')
axes[2].set_ylabel('E(t) / E(0)')
axes[2].set_title('Long-time behavior')
axes[2].legend()

Whoa!  Symplectic Euler's energy isn't perfect.  It wobbles up and down by about 5%.  But it *never drifts*.  After 160 swings it's every bit as good as it was after one.

That's kind of shocking.  RK2 is a second-order method and symplectic Euler is only first order, but over a long run symplectic Euler wins!  (To be fair, RK2 does better for the first few hundred time units.  But its energy leak never stops, so eventually it always loses.)

In phase space, symplectic Euler's orbit closes up on itself.  It isn't quite the exact circle; it's a very slightly squashed and tilted ellipse.  But it goes around that same ellipse over and over instead of spiraling away.

What's going on?

# What Does *Symplectic* Mean?

To understand why swapping two lines helps so much, we need to stop thinking about one trajectory at a time and think about *lots* of them at once.  Good thing we learned how to integrate a whole batch of initial conditions at once!

## A Disk of Initial Conditions

Let's start from a whole disk of states: every $(x, v)$ with energy $E \leq \frac{1}{2}$, which (with $\omega = 1$) is the unit disk in phase space.

We don't need to follow every single state inside the disk.  The edge tells us the shape and the size, so let's just follow lots of states around the edge.  That's a batch with shape `(2, N)`, exactly like last time.

In [ ]:
angle = np.linspace(0, 2*np.pi, 1000, endpoint=False)
disk_edge = np.array([np.cos(angle), np.sin(angle)])   # row 0 is x, row 1 is v
print(disk_edge.shape)

disk_steps = 100   # ten time units

t_disk, euler_disk      = EulerMethod(oscillator, Delta_t, disk_steps).starting_from(0, disk_edge)
t_disk, rk2_disk        = RungeKutta2(oscillator, 0.5, Delta_t, disk_steps).starting_from(0, disk_edge)
t_disk, symplectic_disk = SymplecticEuler(oscillator, Delta_t, disk_steps).starting_from(0, disk_edge)
print(euler_disk.shape)

Let's draw the disk every 2.5 time units.  To see how it spins, we'll also mark the state that started at $(x, v) = (1, 0)$ with a dot.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex='row', sharey='row')

for axis, disk, name in zip(axes, (euler_disk, symplectic_disk), ('Euler', 'symplectic Euler')):
    for step in range(0, disk_steps+1, 25):
        line, = axis.fill(disk[step, 0], disk[step, 1], alpha=0.15)
        axis.plot(disk[step, 0, 0], disk[step, 1, 0], marker='o', color=line.get_facecolor(), alpha=1)
        axis.plot(disk[step, 0], disk[step, 1], color=line.get_facecolor(), alpha=1, label=f't = {t_disk[step]:.1f}')
    axis.set_title(name)
    axis.set_aspect('equal')
    axis.set_xlabel('x')
    axis.set_ylabel('v')

axes[0].legend()

Euler's disk grows and grows.  Symplectic Euler's disk hardly changes at all.  (It actually gets very slightly squashed into a tilted ellipse and back, too slightly to see in detail at this step size.)  Let's measure!

The [*shoelace formula*](https://en.wikipedia.org/wiki/Shoelace_formula) gives the area of a polygon from its corners $(x_i, v_i)$, listed in order around the edge,
\begin{align}
    \text{area} &= \frac{1}{2} \left| \sum_i \left( x_i v_{i+1} - x_{i+1} v_i \right) \right|
\end{align}
where the corner after the last one is the first one again.  `np.roll(v, -1, axis=-1)` shifts every corner over by one (wrapping around at the end), which is exactly the $v_{i+1}$ we need.  Since we do the sum along the last axis, the function computes the area at *every* time at once.

In [ ]:
def area(x, v):
    # The shoelace formula for the area of the polygon with corners (x[i], v[i]), in order around the edge.
    return 0.5 * np.abs(np.sum(x * np.roll(v, -1, axis=-1) - np.roll(x, -1, axis=-1) * v, axis=-1))

euler_area      = area(euler_disk[:, 0],      euler_disk[:, 1])
rk2_area        = area(rk2_disk[:, 0],        rk2_disk[:, 1])
symplectic_area = area(symplectic_disk[:, 0], symplectic_disk[:, 1])

print(euler_area.shape)
print('Euler area grows by a factor of', euler_area[1] / euler_area[0], 'each step')
print('By t = 10, RK2 area has grown by a factor of', rk2_area[-1] / rk2_area[0])

fig, ax = plt.subplots(1,2, figsize=(12, 6))
for i, a in enumerate(ax):
    if i == 0:
        a.plot(t_disk, euler_area / euler_area[0], color='C0', label='Euler')
    a.plot(t_disk, rk2_area / rk2_area[0], color='C1', label='RK2')
    a.plot(t_disk, symplectic_area / symplectic_area[0], color='C2', label='symplectic Euler', linestyle='dashed')
    a.set_xlabel('t')
    a.set_ylabel('area(t) / area(0)')
    a.legend()

Euler's disk grows every single step, by exactly 1% when $\Delta t = 0.1$.  RK2's grows too, just much more slowly.  But symplectic Euler's area doesn't change *at all*.

## The Determinant

For the harmonic oscillator $a = -\omega^2 x$, so one Euler step is a *linear* map, which means we can write it as a matrix,
\begin{align}
    x_{n+1} &= x_n + v_n \Delta t
    &
    v_{n+1} &= v_n - \omega^2 x_n \Delta t
    &
    &\to
    &
    \begin{pmatrix} x_{n+1} \\ v_{n+1} \end{pmatrix}
    &=
    \begin{pmatrix} 1 & \Delta t \\ -\omega^2 \Delta t & 1 \end{pmatrix}
    \begin{pmatrix} x_n \\ v_n \end{pmatrix}.
\end{align}
Remember from linear algebra that a linear map multiplies *every* area by the absolute value of its determinant.  Here
\begin{align}
    \det \begin{pmatrix} 1 & \Delta t \\ -\omega^2 \Delta t & 1 \end{pmatrix} &= 1 + \omega^2 \Delta t^2,
\end{align}
which is $1.01$ when $\omega = 1$ and $\Delta t = 0.1$.  That's exactly the 1% per step we just measured!

Symplectic Euler is two linear maps in a row.  First the kick changes $v$ but not $x$, and then the drift changes $x$ but not $v$,
\begin{align}
    K &= \begin{pmatrix} 1 & 0 \\ -\omega^2 \Delta t & 1 \end{pmatrix}
    &
    D &= \begin{pmatrix} 1 & \Delta t \\ 0 & 1 \end{pmatrix}.
\end{align}
Since the kick happens first, one whole step is the product $DK$ (the matrix on the right acts first).  Both have determinant 1, so
\begin{align}
    \det(DK) &= \det D \times \det K = 1.
\end{align}
Each piece is a *shear*.  The kick slides each point up or down by an amount that depends on its $x$, and the drift slides each point left or right by an amount that depends on its $v$.  Shears tilt shapes, but they never change their area.  (Think of pushing the top of a deck of cards sideways: the deck leans over, but it's still the same amount of deck.)

Plain Euler does both updates at once, both using the old state, and that is *not* a pair of shears.

Let's check all of this with numpy.  `@` is matrix multiplication.

In [ ]:
omega = oscillator.omega

euler_step = np.array([[1, Delta_t], [-omega**2 * Delta_t, 1]])
kick       = np.array([[1, 0],       [-omega**2 * Delta_t, 1]])
drift      = np.array([[1, Delta_t], [0, 1]])

print('det(Euler step) =', np.linalg.det(euler_step))
print('det(kick)       =', np.linalg.det(kick))
print('det(drift)      =', np.linalg.det(drift))
print('det(drift @ kick) =', np.linalg.det(drift @ kick))

# One step of the matrix drift @ kick is one step of SymplecticEuler, for every state on the edge of the disk at once.
print(np.allclose(drift @ kick @ disk_edge, symplectic_disk[1]))

## Beyond the Harmonic Oscillator

For other forces, like the pendulum's, a step isn't a linear map anymore, so there's no single matrix.  But we can still ask how a step stretches a *tiny* patch of phase space near some point $(x, v)$.  That's what the *Jacobian* matrix of partial derivatives tells us,
\begin{align}
    J &= \begin{pmatrix}
        \partial x_{n+1} / \partial x_n & \partial x_{n+1} / \partial v_n
        \\
        \partial v_{n+1} / \partial x_n & \partial v_{n+1} / \partial v_n
    \end{pmatrix},
\end{align}
and its determinant is how much tiny areas get scaled.

A kick, $x \to x$ and $v \to v + a(x)\,\Delta t$, has
\begin{align}
    J_\text{kick} &= \begin{pmatrix} 1 & 0 \\ a'(x)\,\Delta t & 1 \end{pmatrix}
    &
    \det J_\text{kick} &= 1
\end{align}
*no matter what the force $a(x)$ is*.  A drift, $x \to x + v\,\Delta t$ and $v \to v$, has determinant 1 too.  By the chain rule, the Jacobian of a kick followed by a drift is the product of their Jacobians, so symplectic Euler has $\det J = 1$ everywhere, for any force that depends on position.

Plain Euler, $x \to x + v\,\Delta t$ and $v \to v + a(x)\,\Delta t$, has
\begin{align}
    J_\text{Euler} &= \begin{pmatrix} 1 & \Delta t \\ a'(x)\,\Delta t & 1 \end{pmatrix}
    &
    \det J_\text{Euler} &= 1 - a'(x)\,\Delta t^2.
\end{align}
For the oscillator $a'(x) = -\omega^2$ and we get our $1 + \omega^2 \Delta t^2$ back.  For the pendulum it depends on where you are in phase space.

You might worry that "tiny patches" means this is only an approximation.  It isn't!  The change-of-variables formula from multivariable calculus says the area of where a region ends up is $\iint |\det J|\, dx\, dv$ over where it started.  If $\det J = 1$ everywhere, then *every* region, however big or oddly shaped, keeps *exactly* its area.

Let's see that in action with something more dramatic than a disk.  Take a little blob of pendulum states centered at $\theta = 1.5$, $v = 0$.  Remember that pendulums with bigger amplitudes take longer to swing.  So the outer edge of the blob falls behind the inner edge, and the blob should get sheared into a long, thin, curved streak.  If symplectic Euler is really symplectic, the area should stay the same anyway.

The streak gets long and skinny, so we'll use lots of points around the edge.

In [ ]:
pendulum = Pendulum(1.0)

blob_angle = np.linspace(0, 2*np.pi, 4000, endpoint=False)
blob_edge = np.array([1.5 + 0.3 * np.cos(blob_angle), 0.3 * np.sin(blob_angle)])

blob_steps = 200   # twenty time units
t_blob, blob = SymplecticEuler(pendulum, Delta_t, blob_steps).starting_from(0, blob_edge)
blob_area = area(blob[:, 0], blob[:, 1])

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
for step in range(0, 70, 10):
    patch, = ax[0].fill(blob[step, 0], blob[step, 1], alpha=0.3)
    ax[0].plot(blob[step, 0], blob[step, 1], color=patch.get_facecolor(), alpha=1, label=f't = {t_blob[step]:.0f}')

for step in range(0, blob_steps+1, 50):
    patch, = ax[1].fill(blob[step, 0], blob[step, 1], alpha=0.3)
    ax[1].plot(blob[step, 0], blob[step, 1], color=patch.get_facecolor(), alpha=1, label=f't = {t_blob[step]:.0f}')

ax[0].set_title('symplectic Euler, short times')
ax[1].set_title('symplectic Euler, long times')

for a in ax:
    a.set_aspect('equal')
    a.set_xlabel('θ')
    a.set_ylabel('dθ/dt')
    a.legend()

print('largest change in area:', np.max(np.abs(blob_area / blob_area[0] - 1)))

The blob gets stretched into a long, curved streak, but its area doesn't change.  (The tiny leftover change, less than a millionth, is because we're approximating the curved edge with 4000 straight segments.)

An integrator step that preserves phase-space area is called **symplectic**.  (When there's one position and one velocity, like this case, that's the whole story.  With more positions and velocities, *symplectic* is a stronger condition than just preserving volume.)

The *real* motion preserves phase-space area too, for any force that depends only on position.  (That's called [*Liouville's theorem*](https://en.wikipedia.org/wiki/Liouville%27s_theorem_(Hamiltonian)), and is often studied in statistical mechanics.)  So symplectic Euler shares this deep property with the true physics, while plain Euler doesn't.

The mathematician and physicist Hermann Weyl coined the term in 1939, from Greek for "woven together".

## Area and Energy

For the harmonic oscillator, area and energy are very directly connected.  The orbit with energy $E$ is the ellipse
\begin{align}
    \frac{1}{2} v^2 + \frac{1}{2} \omega^2 x^2 &= E
\end{align}
which reaches out to $x = \pm\sqrt{2E}/\omega$ and up to $v = \pm\sqrt{2E}$.  An ellipse with those semi-axes has area $\pi \times \frac{\sqrt{2E}}{\omega} \times \sqrt{2E} = 2\pi E/\omega$.

So for the harmonic oscillator, the area an orbit encloses *is* its energy (times $2\pi/\omega$), and growing area and growing energy are literally the same thing.  When Euler's disk grew by 1% each step, every state on its edge gained 1% energy each step.

For other forces, bigger orbits still enclose more area, but the relationship with the energy isn't a simple proportion.

# The Shadow Energy

If symplectic Euler preserves area, and area is energy for the oscillator, why does its energy wobble by 5%?

Look back at the disk.  Symplectic Euler doesn't carry states around the true circles; it carries them around very slightly squashed, tilted ellipses.  Each state stays on its own tilted ellipse forever.  But a tilted ellipse doesn't have constant $E$: it pokes outside the circle in some places and dips inside it in others.  So $E$ wobbles.

In fact, with a page of algebra you can check that symplectic Euler *exactly* conserves
\begin{align}
    \tilde{E} &= \frac{1}{2} v^2 + \frac{1}{2} \omega^2 x^2 - \frac{1}{2} \omega^2 \Delta t\, x v.
\end{align}
(Try it!  Plug in one kick and one drift and show $\tilde{E}$ comes out the same.)  It's the true energy plus a small correction proportional to $\Delta t$, a kind of *shadow energy*, and its level curves are exactly those tilted ellipses.

In [ ]:
x, v = symplectic_y[:, 0], symplectic_y[:, 1]
shadow_energy = symplectic_energy - 0.5 * omega**2 * Delta_t * x * v

print('       E ranges over', np.ptp(symplectic_energy))
print('shadow E ranges over', np.ptp(shadow_energy))

fig, ax = plt.subplots()
ax.plot(t, symplectic_energy / symplectic_energy[0], color='C2', label='E')
ax.plot(t, shadow_energy / shadow_energy[0], color='black', label='shadow energy')
ax.set_xlim(0, 30)
ax.set_xlabel('t')
ax.set_ylabel('energy(t) / energy(0)')
ax.legend()

The true energy wobbles, but the shadow energy is constant up to roundoff, about $10^{-15}$.

Here's the amazing general fact, which is a theorem we won't prove.  For *any* force that comes from a potential energy $V(x)$, symplectic Euler nearly conserves a shadow energy
\begin{align}
    \tilde{E} &= E - \frac{1}{2} \Delta t\, V'(x)\, v + \left(\text{terms with } \Delta t^2, \Delta t^3, \ldots\right).
\end{align}
(For the oscillator $V'(x) = \omega^2 x$ and the extra terms all vanish, which is why it worked out exactly.)  For other forces the precise statement is a bit subtle: the energy error stays small for an extraordinarily long time.  Not literally forever, but far longer than any simulation you'll ever run.

Area preservation is the reason this shadow energy exists.  RK2 isn't symplectic, it has no shadow energy, and so nothing stops its energy error from creeping along forever.

# Velocity Verlet

Symplectic Euler's energy wobble is 5%, and that's because it's only a first-order method.  Can we get second order *and* keep the area?  Yes!

Split the kick into two halves and put the drift in the middle:
\begin{align}
    v_{n+\frac{1}{2}} &= v_n + \tfrac{1}{2} a(t_n, x_n)\, \Delta t  && \text{(half kick)}
    \\
    x_{n+1} &= x_n + v_{n+\frac{1}{2}}\, \Delta t  && \text{(drift)}
    \\
    v_{n+1} &= v_{n+\frac{1}{2}} + \tfrac{1}{2} a(t_{n+1}, x_{n+1})\, \Delta t  && \text{(half kick)}
\end{align}
Every piece is a kick or a drift, so every piece is a shear with determinant 1, so the whole step is symplectic.  And because the step is symmetric (kick-drift-kick looks the same run backward in time), the first-order errors from the two half kicks cancel, and the method is second order.

This is called *velocity Verlet*, after Loup Verlet, who used it to simulate molecules in 1967.  You'll also see it called *leapfrog*.  It's the workhorse for simulating molecules, planets, and even whole galaxies.

In [ ]:
class VelocityVerlet(FirstOrderODEIntegrator):
    '''
    Velocity Verlet splits each step into a half kick, a drift, and another half kick,

    v_{n+1/2} = v_n       + 1/2 a(t_n, x_n) Δt
    x_{n+1}   = x_n       + v_{n+1/2} Δt
    v_{n+1}   = v_{n+1/2} + 1/2 a(t_{n+1}, x_{n+1}) Δt

    It needs the ODE's acceleration, so it only works on second-order ODEs.
    '''

    def __init__(self, ODE, Delta_t, steps):

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):

        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array with room for a whole state at each time.
        y = np.zeros((self.steps+1,) + np.shape(y_0))

        # The user picked a starting state
        y[0] = y_0

        for step in range(self.steps):
            x, v = y[step]
            # (acceleration also wants a velocity; we hand it the newest one we have.
            #  For forces that depend only on position it doesn't matter.)
            v_half = v      + 0.5 * self.ODE.acceleration(t[step],   x,     v)      * self.Delta_t   # half kick
            x_new  = x      +       v_half                                           * self.Delta_t   # drift
            v_new  = v_half + 0.5 * self.ODE.acceleration(t[step+1], x_new, v_half) * self.Delta_t   # half kick
            y[step+1] = np.array([x_new, v_new])

        return t, y

In [ ]:
t, verlet_y = VelocityVerlet(oscillator, Delta_t, steps).starting_from(0, y_0)
verlet_energy = oscillator.energy(verlet_y[:, 0], verlet_y[:, 1])

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].plot(t, rk2_energy / rk2_energy[0], color='C1', label='RK2')
ax[0].plot(t, symplectic_energy / symplectic_energy[0], color='C2', label='symplectic Euler')
ax[0].plot(t, verlet_energy / verlet_energy[0], color='C3', label='velocity Verlet')
ax[0].set_xlabel('t')
ax[0].set_ylabel('E(t) / E(0)')
ax[0].set_title('Long-time behavior')
ax[0].legend()

ax[1].plot(t[:early], symplectic_energy[:early] / symplectic_energy[0], color='C2', label='symplectic Euler')
ax[1].plot(t[:early], verlet_energy[:early] / verlet_energy[0], color='C3', label='Verlet / Leapfrog')
ax[1].set_xlabel('t')
ax[1].set_ylabel('E(t) / E(0)')
ax[1].set_title('Short-time behavior (t ≤ 10)')
ax[1].legend()

Verlet's energy wobbles too, but only by about 0.25%, twenty times less than symplectic Euler's.  And just like symplectic Euler, it never drifts.

Let's double-check that it preserves the area of our disk, too.

In [ ]:
t_disk, verlet_disk = VelocityVerlet(oscillator, Delta_t, disk_steps).starting_from(0, disk_edge)
verlet_area = area(verlet_disk[:, 0], verlet_disk[:, 1])

print('largest change in area:', np.max(np.abs(verlet_area / verlet_area[0] - 1)))

## Checking the Order

Being symplectic doesn't make a method more *accurate* step by step.  Let's integrate to $t=10$ with smaller and smaller steps and compare the final position with the exact solution, $x(10) = \cos 10$.

In [ ]:
DELTA_T = 1.0 / 2**np.arange(3, 11)
INTEGRATORS = {
    'Euler':            lambda Delta_t, steps: EulerMethod(oscillator, Delta_t, steps),
    'RK2':              lambda Delta_t, steps: RungeKutta2(oscillator, 0.5, Delta_t, steps),
    'symplectic Euler': lambda Delta_t, steps: SymplecticEuler(oscillator, Delta_t, steps),
    'velocity Verlet':  lambda Delta_t, steps: VelocityVerlet(oscillator, Delta_t, steps),
}

fig, ax = plt.subplots()

for color, (name, make_integrator) in zip(('C0', 'C1', 'C2', 'C3'), INTEGRATORS.items()):
    error = np.zeros_like(DELTA_T)
    for i, step_size in enumerate(DELTA_T):
        _, y = make_integrator(step_size, round(10 / step_size)).starting_from(0, y_0)
        error[i] = np.abs(y[-1, 0] - np.cos(10))
    ax.plot(DELTA_T, error, marker='o', linestyle='none', color=color, label=name)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Δt')
ax.set_ylabel('Error |numerical - exact| at $t=10$')
ax.set_title('Error scaling')
ax.legend()

On the log-log plot, Euler and symplectic Euler both have slope 1: they're first order.  RK2 and velocity Verlet both have slope 2: they're second order.

So symplectic integrators aren't magic.  What being symplectic buys you is good behavior over *long* times: no slow leak of energy.

# The Pendulum

So far all our evidence comes from the harmonic oscillator, which is linear and kind of special.  Does any of this survive for a nonlinear force?

Let's release a pendulum from $\theta_0 = 2$ (about 115°) and run all four integrators to $t=1000$ with $\Delta t = 0.1$.

In [ ]:
pendulum = Pendulum(1.0)
theta_start = np.array([2.0, 0.0])

t, pendulum_euler_y      = EulerMethod(pendulum, Delta_t, steps).starting_from(0, theta_start)
t, pendulum_rk2_y        = RungeKutta2(pendulum, 0.5, Delta_t, steps).starting_from(0, theta_start)
t, pendulum_symplectic_y = SymplecticEuler(pendulum, Delta_t, steps).starting_from(0, theta_start)
t, pendulum_verlet_y     = VelocityVerlet(pendulum, Delta_t, steps).starting_from(0, theta_start)

def energy_ratio(y):
    energy = pendulum.energy(y[:, 0], y[:, 1])
    return energy / energy[0]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].plot(t, energy_ratio(pendulum_euler_y), color='C0', label='Euler')
axes[0].plot(t, energy_ratio(pendulum_rk2_y),   color='C1', label='RK2')
axes[0].set_yscale('log')

axes[1].plot(t, energy_ratio(pendulum_rk2_y),        color='C1', label='RK2')
axes[1].plot(t, energy_ratio(pendulum_symplectic_y), color='C2', label='symplectic Euler')
axes[1].plot(t, energy_ratio(pendulum_verlet_y),     color='C3', label='velocity Verlet')

axes[2].plot(t[:early], energy_ratio(pendulum_rk2_y[:early]),        color='C1', label='RK2')
axes[2].plot(t[:early], energy_ratio(pendulum_symplectic_y[:early]), color='C2', label='symplectic Euler')
axes[2].plot(t[:early], energy_ratio(pendulum_verlet_y[:early]),     color='C3', label='velocity Verlet')

for axis in axes:
    axis.set_xlabel('t')
    axis.set_ylabel('E(t) / E(0)')
    axis.legend()

axes[0].set_title('Long-time Energy (log scale)')
axes[1].set_title('Long-time Energy')
axes[2].set_title('Short-time Energy (t ≤ 10)')

Euler gains so much energy (a factor of about 35) that it kicks the pendulum over the top, and then it spins around and around.  RK2 creeps up about 8% by the end.  Symplectic Euler and velocity Verlet both wobble, but they stay put.

For the pendulum $V(\theta) = \omega^2 (1 - \cos\theta)$, so $V'(\theta) = \omega^2 \sin\theta$, and the shadow energy formula says symplectic Euler should nearly conserve $E - \frac{1}{2}\Delta t\, \omega^2 \sin\theta\, v$.  It's not exact anymore, since we dropped the $\Delta t^2$ and higher terms, but it should be *much* steadier than $E$.

In [ ]:
x, v = pendulum_symplectic_y[:, 0], pendulum_symplectic_y[:, 1]
energy = pendulum.energy(x, v)
shadow_energy = energy - 0.5 * Delta_t * pendulum.omega**2 * np.sin(x) * v

print('E        ranges over', np.ptp(energy / energy[0]))
print('shadow E ranges over', np.ptp(shadow_energy / shadow_energy[0]))

The shadow energy wobbles about 70 times less than the energy itself.  The general theory works for nonlinear forces too.

# The Fine Print

## Symplectic Doesn't Mean Exact

Velocity Verlet holds onto the energy beautifully.  But is it getting the *motion* right?  Let's compare $x(t)$ with the exact solution near the end of our long harmonic oscillator run.

In [ ]:
exact = oscillator.exact_solution(t, 1.0, 0.0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, verlet_y[:, 0], color='C3', label='velocity Verlet Δt=0.1')
ax.plot(t, exact[0], color='black', label='exact')

# Compare with a finer integration
fine_t, fine_verlet_y = VelocityVerlet(oscillator, Delta_t/10, 10*steps).starting_from(0, y_0)
ax.plot(fine_t, fine_verlet_y[:, 0], color='C4', linestyle='dotted', linewidth=4, label='velocity Verlet Δt=0.01')

ax.set_xlim(980, 1000)
ax.set_xlabel('t')
ax.set_ylabel('x')
ax.legend()

Verlet's oscillator runs a little bit *fast* with $\Delta t = 0.1$.  Its orbit has the right shape and the right size, but it goes around slightly too quickly, so by $t = 1000$ it's ahead of the true solution by about 0.4 radians, a sizable chunk of a swing.  But with a smaller step size $\Delta t=0.01$ the integrator is reliable out this far, at least by eye.

Symplectic integrators get the *shape* of the orbit right for a very long time.  They don't stop *timing* errors from piling up.  If you care where a planet will be on a particular date, you still need small steps!

## Which Forces?

These methods shine when the force depends only on position, like gravity, springs, or the pendulum.

With drag there's no conserved energy to protect.  A real system with friction *loses* energy, and its blobs of phase space really do shrink.  So symplectic methods lose their edge.  (Velocity Verlet even gets awkward: its second half kick would need the force at the *new* velocity, which we haven't finished computing yet.)  Magnetic forces need their own special tricks, too.

A force that changes with time, like a periodic push, is fine.  The kicks just use $a(t, x)$, and each step still preserves area.  The energy won't be conserved, but that's real physics: the push pumps energy in and out.

The main takeaway is that the best integrator isn't necessarily the most accurate one step-by-step.  It's one that respects the physics!